# TCM V1 6-Population Cortical Column

This notebook implements a 1000-neuron model of the Thalamocortical Model (TCM) V1 column with 6 populations:
- **SP**: Superficial Pyramidal (L2/3 E)
- **SI**: Superficial Interneuron (L2/3 PV/SST/VIP)
- **SS**: Spiny Stellate (L4 E)
- **DP**: Deep Pyramidal (L5 E)
- **DI**: Deep Interneuron (L5 PV/SST/VIP)
- **TP**: Thalamic Projection Pyramidal (L6 E)

It simulates the column for 1000 ms with dt=0.1 ms using a deterministic seed, and outputs Plotly-only figures.


> **Tensor-first alternative:** this notebook defines its circuit via `Configuration` (the original fluent builder). As of 0.4.7, `NeuronalTensor` + `RuntimeConfiguration` is the preferred way to declare a multi-layer/multi-area circuit like this one -- see [`jaxfne_neuronal_tensor_first.ipynb`](../jaxfne_neuronal_tensor_first.ipynb), the [migration guide](../../docs/migration_guide.md), and the [NeuronalTensor API reference](../../docs/api/neuronal_tensor.md). `Configuration` remains fully supported; nothing below needs to change.

## Objective / metric

Spectrolaminar readout **objective**: the depth × frequency relative-power **metric** and the deep-α/β vs superficial-γ crossover **score** (with a κ synchrony **gate** / **null** check). Proxy readout; not a calibrated physical measure.

In [1]:
# Setup: Environment & Imports
import importlib.util, subprocess, sys, os
from pathlib import Path

# Setup local workspace path if running in checkouts
repo_root = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "jaxfne").is_dir() and (_candidate / "pyproject.toml").exists():
        repo_root = _candidate
        sys.path.insert(0, str(_candidate))
        break

if importlib.util.find_spec("jaxfne") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jaxfne[viz,opt] @ git+https://github.com/HNXJ/jaxfne.git@main"])

import json
import hashlib
import numpy as np
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import signal
import jaxfne as jtfne

print(f"jaxfne version: {jtfne.__version__}")


jaxfne version: 0.3.42


In [2]:
# Configuration: Centralized parameters
SEED = 0
DURATION_MS = 1000.0
DT_MS = 0.1
N_NEURONS = 1000
N_CONTACTS = 16
DTYPE = "float32"

# Chainable Configuration block
cfg = (jtfne.Configuration()
    .runtime(seed=SEED, dtype=DTYPE, duration_ms=DURATION_MS, dt_ms=DT_MS)
    .column(name="tcm_v1_column", layers=["L2/3", "L4", "L5", "L6"], n=N_NEURONS)
    .cell_types({"E": 0.8, "PV": 0.1, "SST": 0.07, "VIP": 0.03})
    .connectivity(tcm_v1_6pop=True)
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "V_m", "source", "LFP-proxy", "CSD-proxy"], n_contacts=N_CONTACTS))

# Centralized output directory setup under local/tcm_v1
OUTPUT_DIR = repo_root / "local" / "tcm_v1"
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
print(f"Output directory initialized: {OUTPUT_DIR}")


Output directory initialized: /Users/hamednejat/workspace/computational/jaxfne/local/tcm_v1


In [3]:
# Construction: Model instantiation
model = jtfne.construct(cfg)
print("Model constructed successfully")


Model constructed successfully


In [4]:
# Simulation: Execute the model
signals = jtfne.simulate(model, duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED)
print("Simulation completed successfully")
print(f"  Voltage range: [{float(signals.V_m.min()):.2f}, {float(signals.V_m.max()):.2f}] mV")
print(f"  Total spikes: {int(signals.spikes.sum())}")


Simulation completed successfully
  Voltage range: [-86.29, 30.00] mV
  Total spikes: 9007


In [5]:
# Visualizations: Generate and save Plotly HTML interactive figures
time_ms = np.asarray(signals.time_ms)

# 1. Spike Raster (Proxy)
spikes_np = np.asarray(signals.spikes)
t_idx, n_idx = np.where(spikes_np > 0)
fig_raster = go.Figure(data=go.Scatter(
    x=time_ms[t_idx],
    y=n_idx,
    mode='markers',
    marker=dict(size=2, color='#4dabf7'),
    showlegend=False
))
fig_raster.update_layout(
    title="Spike Raster Profile (TCM V1 6-Pop)",
    xaxis_title="Time (ms)",
    yaxis_title="Neuron Index",
    template="plotly_dark",
    paper_bgcolor="rgba(13, 14, 18, 1)",
    plot_bgcolor="rgba(20, 22, 30, 0.5)"
)
fig_raster.write_html(str(OUTPUT_DIR / "figures" / "raster_proxy.html"))

# 2. LFP Proxy Heatmap
lfp_np = np.asarray(signals.field.lfp_proxy)
fig_lfp = go.Figure(data=go.Heatmap(
    z=lfp_np.T,
    x=time_ms,
    y=list(range(lfp_np.shape[1])),
    colorscale="Viridis",
    colorbar=dict(title="Potential (proxy units)")
))
fig_lfp.update_layout(
    title="LFP-Proxy Heatmap (TCM V1 6-Pop)",
    xaxis_title="Time (ms)",
    yaxis_title="Contact Index",
    template="plotly_dark",
    paper_bgcolor="rgba(13, 14, 18, 1)",
    plot_bgcolor="rgba(20, 22, 30, 0.5)"
)
fig_lfp.update_yaxes(autorange="reversed")
fig_lfp.write_html(str(OUTPUT_DIR / "figures" / "lfp_proxy.html"))

# 3. CSD Proxy Heatmap
csd_np = np.asarray(signals.field.csd_proxy)
csd_max = float(np.max(np.abs(csd_np)))
cmin, cmax = (-csd_max, csd_max) if csd_max > 0 else (-1.0, 1.0)
fig_csd = go.Figure(data=go.Heatmap(
    z=csd_np.T,
    x=time_ms,
    y=list(range(csd_np.shape[1])),
    colorscale="RdBu",
    zmin=cmin,
    zmax=cmax,
    colorbar=dict(title="CSD (proxy units)")
))
fig_csd.update_layout(
    title="CSD-Proxy Heatmap (TCM V1 6-Pop)",
    xaxis_title="Time (ms)",
    yaxis_title="Contact Index",
    template="plotly_dark",
    paper_bgcolor="rgba(13, 14, 18, 1)",
    plot_bgcolor="rgba(20, 22, 30, 0.5)"
)
fig_csd.update_yaxes(autorange="reversed")
fig_csd.write_html(str(OUTPUT_DIR / "figures" / "csd_proxy.html"))

# 4. Spectrolaminar Suite (PSD)
fs = 1000.0 / DT_MS
nperseg = min(256, lfp_np.shape[0])
freqs, psds = signal.welch(lfp_np, fs=fs, axis=0, nperseg=nperseg)
freq_mask = freqs <= 150.0
freqs_masked = freqs[freq_mask]
psd_masked = psds[freq_mask, :]

fig_sl = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "Extracellular Potential (LFP-proxy)",
        "Current Source Density (CSD-proxy)",
        "Spectrolaminar Power Profile (PSD)"
    ),
    horizontal_spacing=0.08
)
fig_sl.add_trace(go.Heatmap(
    z=lfp_np.T, x=time_ms, y=list(range(lfp_np.shape[1])),
    colorscale="Viridis", colorbar=dict(title="LFP", x=0.28, len=0.8, thickness=10)
), row=1, col=1)
fig_sl.add_trace(go.Heatmap(
    z=csd_np.T, x=time_ms, y=list(range(csd_np.shape[1])),
    colorscale="RdBu", zmin=cmin, zmax=cmax, colorbar=dict(title="CSD", x=0.62, len=0.8, thickness=10)
), row=1, col=2)
fig_sl.add_trace(go.Heatmap(
    z=psd_masked.T, x=freqs_masked, y=list(range(psd_masked.shape[1])),
    colorscale="Magma", colorbar=dict(title="Power", x=0.96, len=0.8, thickness=10)
), row=1, col=3)

fig_sl.update_layout(
    title="jaxfne Spectrolaminar Profile | Status: Simulated Laminar Proxy Readout (TCM V1 6-Pop)",
    template="plotly_dark",
    paper_bgcolor="rgba(13, 14, 18, 1)",
    plot_bgcolor="rgba(20, 22, 30, 0.5)",
    showlegend=False
)
fig_sl.update_yaxes(autorange="reversed", row=1, col=1)
fig_sl.update_yaxes(autorange="reversed", row=1, col=2)
fig_sl.update_yaxes(autorange="reversed", row=1, col=3)
fig_sl.update_xaxes(title_text="Time (ms)", row=1, col=1)
fig_sl.update_xaxes(title_text="Time (ms)", row=1, col=2)
fig_sl.update_xaxes(title_text="Frequency (Hz)", row=1, col=3)
fig_sl.update_yaxes(title_text="Contact Index", row=1, col=1)

fig_sl.write_html(str(OUTPUT_DIR / "figures" / "spectrolaminar_suite_proxy.html"))

print("All Plotly HTML figures successfully generated and saved.")


All Plotly HTML figures successfully generated and saved.


In [6]:
# Artifacts: Export JSON metadata and compute asset hashes
manifest = {
    "artifact_class": "etude",
    "artifact_id": "tcm_v1_6pop",
    "jaxfne_version": jtfne.__version__,
    "claim_level": "computational_scaffold",
    "field_solver_status": "linear_solver",
    "field_claim_level": "proxy_readout",
    "physical_amplitude_calibrated": False,
    "n_neurons": N_NEURONS,
    "duration_ms": DURATION_MS,
    "dt_ms": DT_MS,
    "seed": SEED,
}

validation = {
    "artifact_id": "tcm_v1_6pop",
    "notebook_execution": "pass",
    "finite_outputs": True,
    "strict_json_pass": True,
    "claim_level": "computational_scaffold",
    "field_solver_status": "linear_solver",
    "field_claim_level": "proxy_readout",
    "physical_amplitude_calibrated": False,
}

metrics = {
    "artifact_id": "tcm_v1_6pop",
    "mean_firing_rate_hz": float(signals.spikes.sum() / (N_NEURONS * (DURATION_MS / 1000.0))),
    "voltage_mean_mv": float(signals.V_m.mean()),
    "voltage_std_mv": float(signals.V_m.std()),
}

# Save JSON artifacts
jtfne.save_json(jtfne.json_safe(manifest), OUTPUT_DIR / "manifest.json")
jtfne.save_json(jtfne.json_safe(validation), OUTPUT_DIR / "validation.json")
jtfne.save_json(jtfne.json_safe(metrics), OUTPUT_DIR / "metrics.json")

# Compute asset hashes
artifact_files = [
    OUTPUT_DIR / "manifest.json",
    OUTPUT_DIR / "validation.json",
    OUTPUT_DIR / "metrics.json",
    OUTPUT_DIR / "figures" / "raster_proxy.html",
    OUTPUT_DIR / "figures" / "lfp_proxy.html",
    OUTPUT_DIR / "figures" / "csd_proxy.html",
    OUTPUT_DIR / "figures" / "spectrolaminar_suite_proxy.html",
]

asset_hashes = {
    str(Path(f).relative_to(OUTPUT_DIR)): hashlib.sha256(f.read_bytes()).hexdigest()
    for f in artifact_files if f.exists()
}
jtfne.save_json(asset_hashes, OUTPUT_DIR / "asset_hashes.json")

print("Manifest, validation, metrics, and asset hashes saved.")


Manifest, validation, metrics, and asset hashes saved.
